In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Data path
file_path = "../Data/cleaned_data.csv" 
df = pd.read_csv(file_path)

FileNotFoundError: [Errno 2] No such file or directory: '../Data/cleaned_data.csv'

In [ ]:
# Remove unnecessary columns: 'Series Code' and 'Country Code'
# These columns do not provide useful information for our analysis. They are identifiers that are redundant and not needed for modeling.
df = df.drop(columns=['Series Code', 'Country Code'], errors='ignore')

In [ ]:
# Standardize column names
# We converted these column names to lowercase, replaced spaces with underscores, and removed special characters to ensures consistency.
df.columns = (df.columns
              # remove trailing spaces
              .str.strip()
              # convert to lowercase
              .str.lower() 
              # replace spaces with underscores
              .str.replace(' ', '_')  
              # remove special characters
              .str.replace(r'[^a-zA-Z0-9_]', '', regex=True))  
# check 
df.head()

In [ ]:
# Check for missing values
# We identified missing data before reshaping, which will helps us decide how to handle missing values in later preprocessing steps.
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values

In [ ]:
# Reshape the dataset to long format. 
# We are converting the data to long format, where each row represents a single year for each country. This makes it easier to filter and manipulate the data, ensures compatibility with time-series forecasting models, and simplifies trend analysis.
# Columns containing 'yr' in their names
year_columns = [col for col in df.columns if 'yr' in col]

# Convert
df_long = df.melt(id_vars=['series_name', 'country_name', 'category', 'income_group'], 
                  value_vars=year_columns, 
                  var_name='year', value_name='value')

# Convert 'year' to integer
df_long['year'] = df_long['year'].str.extract('(\d+)').astype(int)

# Check results
df_long.head()
df_long.columns

In [ ]:
# Split dataset into 70% training, 15% validation, and 15% test Sets
train_df = df_long[df_long["year"].between(2002, 2015)]  # 2002-2015
val_df = df_long[df_long["year"].between(2016, 2018)]    # 2016-2018
test_df = df_long[df_long["year"].between(2019, 2021)]   # 2019-2021

In [ ]:
# Save the datasets
train_df.to_csv("/mnt/data/train_data.csv", index=False)
val_df.to_csv("/mnt/data/val_data.csv", index=False)
test_df.to_csv("/mnt/data/test_data.csv", index=False)

In [ ]:
# Load the training data
train_path = "/mnt/data/train_data.csv"
train_df = pd.read_csv(train_path)

In [ ]:
# Find variables that contain zero values
zero_value_counts = train_df[train_df["value"] == 0].groupby("series_name").size()
zero_value_counts

In [ ]:
# Variables where zero values should be replaced with NaN
replace_zero_indicators = [
    "proportion of population pushed below",
    "risk of catastrophic expenditure",
    "risk of impoverishing expenditure",
    "proportion of population spending more than 25%",
    "pregnant women receiving prenatal care"
]

# Replace
train_df.loc[
    (train_df['value'] == 0) & 
    (train_df['series_name'].str.contains('|'.join(replace_zero_indicators), case=False)),
    'value'
] = np.nan

# Check the results
print(train_df[train_df['value'].isna() & train_df['series_name'].str.contains('|'.join(replace_zero_indicators), case=False)])

For zero values in our dataset, we first identified which indicators contained at least one zero. Several healthcare-related indicators, such as External Health Expenditure (%) and Out-of-Pocket Health Expenditure (%) had zero values. Then, we examined specific rows where the value was 0 to determine whether these zeros represented actual data or missing values. We found that some zeros were meaningful and reflected real values in the dataset, so we kept them. But, in cases where zero values were unlikely to be real, we replaced them with NaN to indicate missing data.
For example,
- For External Health Expenditure (%) → zeros are likely meaningful if a country does not receive external health funding, a value of 0 makes sense. So we should not remove these as they indicate true disparities in how healthcare is funded across countries.
- For Physician & Community Health Workers Per 1,000 People → zeros may be valid if a country lacks medical professionals. We should verify if these zeroes persist over multiple years to confirm.
- For Proportion of Population Spending on Healthcare → zeros might indicate missing data if some countries consistently have 0 values for out-of-pocket healthcare spending, this might be unreported data rather than actual zero spending. We should cross-check with this.
- For Incidence of Malaria & Maternal Mortality Ratio → zeros could be meaningful to some high income countries.

In [ ]:
# Log Transformation
# Calculate skewness
skew_values = df_long.groupby('series_name')['value'].skew().sort_values(ascending=False)

# Top 10 most skewed variables
print(skew_values.head(10))

In [ ]:
# Define variables for log transformation
log_transform_vars = [
    "Hospital beds (per 1,000 people)",
    "Lifetime risk of maternal death (%)",
    "Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70 (%)",
    "Proportion of population spending more than 25% of household consumption or income on out-of-pocket health care expenditure (%)",
    "Poverty headcount ratio at national poverty line (% of population)"
]

# Filter the dataset 
df_before_log = train_df[train_df['series_name'].isin(log_transform_vars)]

# Visualizing each variable before log transformation
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 10))
axes = axes.flatten()
for i, indicator in enumerate(log_transform_vars):
    subset = df_before_log[df_before_log['series_name'] == indicator]
    sns.histplot(subset['value'], bins=20, kde=True, color="steelblue", ax=axes[i])
    axes[i].set_title(indicator)
    axes[i].set_xlabel("Value")
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Apply log transformation only for positive values
train_df.loc[
    (train_df['series_name'].isin(log_transform_vars)) & (train_df['value'] > 0), 
    'value'
] = np.log1p(train_df['value'])

# Visualizing transformed variables
plt.figure(figsize=(12, 8))
for i, indicator in enumerate(log_transform_vars):
    plt.subplot(3, 2, i + 1)
    subset = train_df[train_df['series_name'] == indicator]
    sns.histplot(subset['value'], bins=20, kde=True)
    plt.title(indicator)
    plt.xlabel("Log-transformed value")

plt.tight_layout()
plt.show()

We applied log transformation to handle skewed data distributions and also to improve model performance. In our case, we applied log transformation selectively based on the distribution characteristics of the variables.

Specifically, we applied a log transformation to:
- Hospital beds (per 1,000 people) → some countries have very few hospital beds, but others have much higher values, creating a highly skewed distribution. Log transformation helps normalize the spread.

- Lifetime risk of maternal death (%) → many countries have near-zero maternal mortality, but others have significantly higher risks, which leads to an uneven distribution. Log transformation makes the data more comparable.

- Mortality from CVD, cancer, diabetes, or CRD (%) → the distribution is right-skewed with a few countries experiencing very high mortality rates. Log transformation reduces this skewness.

- Proportion of population spending >25% of income on healthcare (%) → many countries have low out-of-pocket expenditures, but some have much higher burdens. Log transformation improves the scale balance.

- Poverty headcount ratio at the national poverty line (%) → this metric varies widely across income groups, which leads to a skewed distribution. Log transformation stabilizes variance and makes patterns more interpretable.

In [ ]:
# Remove any duplicate rows
df_duplicates = df.drop_duplicates()

# Check how many duplicates were removed
num_removed = len(df) - len(df_duplicates)
print({num_removed})
df_duplicates.head()

We removed duplicate rows for data integrity and prevent misleading results in our analysis. Duplicates can distort our statistical summaries and bias machine learning models by giving extra weight to repeated observations. So we remove exact duplicates ensures that each data point is counted only once and maintaining accuracy in trends and predictions. Since some duplicates in our dataset represented distinct categories like healthcare resources vs. healthcare expenditure, we kept those that added meaningful context and eliminates the true redundancies. This helps us to improve model reliability and fair comparisons across countries and years.

In [ ]:
# Finding outliers
# Identify year columns
year_columns = [col for col in df.columns if '_yr' in col]

# Convert all year columns to numeric
df[year_columns] = df[year_columns].apply(pd.to_numeric, errors='coerce')

# Function to detect and cap outliers for each year column
def cap_outliers_iqr(df, year_columns):
    for col in year_columns:
        # 25th percentile
        Q1 = df[col].quantile(0.25)
        # 75th percentile
        Q3 = df[col].quantile(0.75)
        # Interquartile range
        IQR = Q3 - Q1 
        # Outlier thresholds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Cap outliers
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    return df

# Apply outlier capping function
df_cleaned = cap_outliers_iqr(df, year_columns)

# First  10 row of sesults
print(df_cleaned.head(10))

We handled outliers in the dataset to prevent extreme values from distorting our analysis, which could skew trends and mislead predictive models. Since many machine learning models struggle with extreme values, keeping them unchanged could lead to biased predictions, overfitting, and inaccurate relationships between healthcare investments and life expectancy. So instead of removing them, we used the IQR method to cap extreme values, and ensures that our dataset remains balanced and still preserving meaningful variations.

In [ ]:
# Categorizing countries by income level with numeric encoding 1, 2, 3
# Only map if "income_group" exists
if "income_group" in df.columns:
    income_mapping = {
        "Low-Income Countries": 1,
        "Middle-Income Countries": 2,
        "High-Income Countries": 3
    }
    
    # Assign numeric encoding
    df["income_group_numeric"] = df["income_group"].map(income_mapping)

    # Check for missing values
    df["income_group_numeric"] = df["income_group_numeric"].fillna(0).astype(int)

    # Drop original column
    df = df.drop(columns=["income_group"], errors='ignore')
    
    # Check for year based and non year columns
    year_columns = [col for col in df.columns if "_yr" in col]  
    id_vars = [col for col in df.columns if col not in year_columns]

We categorize countries by income level using numeric encoding (1 = low, 2 = middle, 3 = high) to make the data usable for modeling and analysis. Many models cannot process categorical text, so converting income_group into income_group_numeric allows for effective feature utilization.

In [ ]:
# Reshape data to long format
df_long = df.melt(id_vars=id_vars, value_vars=year_columns, var_name="year_column", value_name="value")

# Extract year from column names
df_long["year"] = df_long["year_column"].str.extract(r'(\d{4})').astype(int)

# Drop "year_column"
df_long = df_long.drop(columns=["year_column"])

# Sort data
df_long = df_long.sort_values(by=["country_name", "series_name", "year"])

# Check if each country has a consistent income group classification
df_long["income_group_numeric"] = df_long.groupby("country_name")["income_group_numeric"].transform("max")

We reshape the training data to long format again to ensure consistency with our preprocessing pipeline as the original reshaped structure may not have been carried forward after splitting.

In [ ]:
# Feature engineering
# Generate lag features for 3 year and 5 year
df_long["lag3"] = df_long.groupby(["country_name", "series_name"])["value"].shift(3)
df_long["lag5"] = df_long.groupby(["country_name", "series_name"])["value"].shift(5)

We created 3 year and 5 year lag features to capture how past healthcare trends influence present outcomes in life expectancy. Since all healthcare investments, policy, and economic shifts take time to show measurable effects, we used lagged features helps the model recognize these delayed impacts. We used 3 year lag captures mid-term effects, and a 5 year lag accounts for longer-term trends. So that our model doesn’t just rely on current year data but understands how past healthcare conditions shape present disparities.

In [ ]:
# Compute rolling averages for 3 year and 5 year
df_long["rolling3"] = df_long.groupby(["country_name", "series_name"])["value"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)
df_long["rolling5"] = df_long.groupby(["country_name", "series_name"])["value"].rolling(5, min_periods=1).mean().reset_index(level=[0,1], drop=True)

Just like how we did with lag features, we created 3 year and 5 year rolling averages to smooth short term fluctuations in healthcare indicators and highlight long term trends. Healthcare data can be noisy due to sudden economic changes or external events like pandemics. So we used a 3 year rolling average to captures mid-term variations, and 5-year rolling average to provide a broader view of sustained trends. This will help our model to focus on meaningful long term healthcare disparities.

In [ ]:
# Compute year over year growth rate
df_long["growth_rate"] = df_long.groupby(["country_name", "series_name"])["value"].pct_change()

We compute the year over year growth rate to measure the relative change in values over time, which will provides insights into trends for each country. This helps identify patterns in healthcare expenditure, resource allocation, and economic indicators, so that it is easier to detect anomalies, growth, and policy impacts. 

In [ ]:
# Fill NaN growth rates with 0
df_long["growth_rate"] = df_long["growth_rate"].fillna(0)

# Fill missing lag values with country series mean
df_long["lag3"] = df_long.groupby(["country_name", "series_name"])["lag3"].transform(lambda x: x.fillna(x.mean()))
df_long["lag5"] = df_long.groupby(["country_name", "series_name"])["lag5"].transform(lambda x: x.fillna(x.mean()))

# Forward fill missing values in rolling averages to maintain smooth trends
df_long["rolling3"] = df_long.groupby(["country_name", "series_name"])["rolling3"].ffill()
df_long["rolling5"] = df_long.groupby(["country_name", "series_name"])["rolling5"].ffill()


After applying all the changes, we found that some columns contained missing values. Specifically, the lag features and rolling averages. To solve this, we used a hybrid approach to maintain data integrity while preserving trends. For rolling averages, we applied forward fill to make sure that missing values were replaced with the last available data point to keep trends smooth. For lag features, we used group wise mean imputation to fill in missing values with the average for each country to prevent data loss without distorting patterns.

In [ ]:
# Save the dataset
output_file_path = "../Data/cleaned_train_dataset.csv"
df_long.to_csv(output_file_path, index=False)
